# NB16 — POSITION-AWARE CNN1D SENSITIVITY, ROW-LEVEL DIAGNOSTIC AND ORDER ABLATION | NIR-HUEVOS 2026

**Post hoc sensitivity analysis for the revision. It does not replace, modify or re-run NB01–NB15.**

## Why this notebook exists
NB11 evaluated one compact CNN1D whose last convolutional block ends in **global average pooling (GAP)**.
GAP discards *where* along the 740–1070 nm axis a learned feature occurred, whereas in NIR regression the
position of an absorption band is usually the information. A reviewer can therefore argue that the negative CNN1D
result reflects that design choice rather than "local convolution" in general. NB16 tests this directly with
**exactly the same frozen protocol as NB11** (same egg-disjoint outer/inner folds, same five preprocessing candidates,
same optimiser, early stopping, epoch rule and final seeds 2026/2027/2028), changing only how positional
information is handled:

| Variant | Description | Trainable params |
|---|---|---|
| `CNN1D` (reference, NB11) | Conv16(k7)-BN-MP2-Conv32(k5)-BN-**GAP**-Dense32 | 3,905 (4,001 total) |
| `CNN1D_GAP_POS` | Same network, plus a fixed wavelength-position input channel | ≈4,017 |
| `CNN1D_FLAT` | Conv16(k7)-BN-MP4-Conv32(k5)-BN-MP4-**Flatten**-Dense32 (position-preserving head) | **23,361 (= ANN)** |

## Parts (switch each one in the first code cell)
* **Part A** — full nested benchmark of `CNN1D_FLAT` and `CNN1D_GAP_POS` (inner CV chooses preprocessing and epochs; outer test eggs are never used for selection). Adds paired egg-level statistics against SVR, PLSR, ANN and the NB11 CNN1D.
* **Part B** — row-level (leaky) partition diagnostic for the three CNN variants, using the *same* row-level folds as NB09A. Extends manuscript Table 4.
* **Part C** — wavelength-order ablation (reversed + 5 shuffles: NB05 seed 52026 and NB05B seeds 11017/24601/73819/90210) for `CNN1D` and `CNN1D_FLAT`, using the exact permutation maps of NB05/NB05B. For a convolutional model that uses local spectral structure, shuffling *should* hurt — this is the positive control the RNN ablation lacks. Extends manuscript Table 6.

## Protocol controls
* Outer-test eggs are used only to obtain out-of-fold predictions. Preprocessing statistics are fitted on training rows only.
* Everything is resumable: each fit is checkpointed in `_CHECKPOINT`. If Colab disconnects, reconnect the GPU and run all cells again.
* Bootstrap: 10,000 whole-egg resamples, seed 20260915 (same as NB12).

## Runtime (Colab GPU) — approximate
Part A ≈ 60–90 min · Part B ≈ 10–15 min · Part C ≈ 60–90 min. A CPU runtime works but is 5–10× slower.
Select *Runtime → Change runtime type → GPU (T4)* before running.

## Output
`05_RESULTS/REVISION_REVIEWERS_2026_09/NB16_CNN1D_POSITION_AWARE_SENSITIVITY/` and a ZIP in `05_RESULTS/ZIP_PACKAGES/`
(downloaded by the last cell). Send that ZIP back for integration into the manuscript.

In [ ]:
import os, sys, json, gc, time, random, hashlib, platform, subprocess, shutil, warnings, re
from pathlib import Path
from datetime import datetime, timezone
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ------------------------------- switches -------------------------------
QUICK_TEST       = os.environ.get('NIR_QUICK_TEST', '0') == '1'   # smoke test only; NEVER use for reported results
STRICT_INTEGRITY = os.environ.get('NIR_STRICT', '1') == '1'       # verify SHA-256 of dataset and frozen split manifest
RUN_PART_A = True
RUN_PART_B = True
RUN_PART_C = True

# ------------------------------- paths ---------------------------------
PROJECT_ROOT = Path(os.environ.get('NIR_PROJECT_ROOT', '/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026'))
RAW_DIR   = PROJECT_ROOT / '01_DATA_RAW'
SPLIT_DIR = PROJECT_ROOT / '03_SPLITS_FROZEN'
RES_ROOT  = PROJECT_ROOT / '05_RESULTS'
RESULT_DIR = RES_ROOT / 'REVISION_REVIEWERS_2026_09' / 'NB16_CNN1D_POSITION_AWARE_SENSITIVITY'
CKPT_DIR   = RESULT_DIR / '_CHECKPOINT'
ZIP_DIR    = RES_ROOT / 'ZIP_PACKAGES'
for p in [RESULT_DIR, CKPT_DIR, ZIP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATA_FILE  = RAW_DIR / 'dataset_egg_storage_RAW.csv'
OUTER_FILE = SPLIT_DIR / 'outer_group_assignment_seed2026.csv'
SPLIT_MANIFEST_FILE = SPLIT_DIR / 'split_manifest.json'
EXPECTED_DATASET_SHA256 = 'cd5021c555ae6b57f892549c574599cef75edf87f58b3f7f4d246ade9327d15e'
EXPECTED_SPLIT_MANIFEST_SHA256 = 'fbeb8fa19d522cd91bee875bf5731cda264475da27bc7e93c25ca0d6f0f33717'

RUN_REVISION = 'NB16_v1_position_aware_cnn1d_sensitivity'

# ------------------------- frozen protocol (= NB11) --------------------
N_OUTER, N_INNER = 5, 4
PREP_ORDER = ['raw', 'snv', 'msc', 'sg_smooth', 'sg_deriv1']
SG_WINDOW, SG_POLYORDER = 11, 2
BATCH_SIZE, LEARNING_RATE, DROPOUT = 32, 1e-3, 0.20
MAX_INNER_EPOCHS, PATIENCE, MIN_DELTA = 250, 20, 0.001
FINAL_SEEDS = [2026, 2027, 2028]
BOOT_REPS, BOOT_SEED = 10000, 20260915
OUTER_FOLDS = list(range(1, N_OUTER + 1))
PREPS = list(PREP_ORDER)
NEW_VARIANTS = ['CNN1D_FLAT', 'CNN1D_GAP_POS']
SEED_BASE = {'CNN1D_FLAT': 62000, 'CNN1D_GAP_POS': 63000}   # inner seed = base + 100*outer + 10*prep_index + inner
REF = 'CNN1D'                                               # NB11 reference (GAP) name used in NB12 files

if QUICK_TEST:
    MAX_INNER_EPOCHS, FINAL_SEEDS, BOOT_REPS = 3, [2026], 300
    PREPS = ['raw', 'sg_deriv1']
    print('*** QUICK_TEST ACTIVE: tiny epochs, 1 seed, 2 preprocessings. Results are NOT valid. ***')

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPUs:', [g.name for g in gpus] or 'none (CPU)')
print('Project root:', PROJECT_ROOT, '| exists:', PROJECT_ROOT.exists())

Mounted at /content/drive
TensorFlow 2.20.0 | Keras 3.13.2 | GPUs: ['/physical_device:GPU:0']
Project root: /content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026 | exists: True


In [ ]:
# ---------------------------- integrity gate ----------------------------
def sha256_file(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def find_one(filename, root=PROJECT_ROOT, required=True):
    hits = sorted(root.rglob(filename))
    if not hits:
        assert not required, f'Not found anywhere under {root}: {filename}'
        return None
    if len(hits) > 1:
        print(f'  note: {len(hits)} copies of {filename}; using {hits[0].relative_to(root)}')
    return hits[0]

for p in [DATA_FILE, OUTER_FILE, SPLIT_MANIFEST_FILE]:
    assert p.exists(), f'Missing required input: {p}'
dataset_sha = sha256_file(DATA_FILE)
split_sha = sha256_file(SPLIT_MANIFEST_FILE)
if STRICT_INTEGRITY:
    assert dataset_sha == EXPECTED_DATASET_SHA256, 'Dataset hash changed.'
    assert split_sha == EXPECTED_SPLIT_MANIFEST_SHA256, 'Frozen split manifest hash changed.'
    manifest = json.loads(SPLIT_MANIFEST_FILE.read_text(encoding='utf-8'))
    for fname, expected in manifest['files'].items():
        fp = SPLIT_DIR / fname
        assert fp.exists() and sha256_file(fp) == expected, f'Frozen split changed: {fname}'
    print('PASS - dataset and frozen splits verified by SHA-256.')
else:
    print('WARNING: integrity gate skipped (STRICT_INTEGRITY=False).')

# reference inputs produced by earlier notebooks (searched by name; no hard-coded sub-folders)
NB11_SELECTED_FILE = find_one('NB11_selected_configurations.csv')
NB11_SEEDWISE_FILE = find_one('NB11_oof_predictions_seedwise.csv')
NB12_PEREGG_FILE   = find_one('NB12_per_egg_MAE_wide.csv')
NB12_UNIFIED_FILE  = find_one('NB12_unified_oof_predictions.csv')
NB09A_ROWLEVEL_FILE = find_one('NB09A_rowlevel_oof_seedmean.csv', required=RUN_PART_B)
NB09A_COMPARISON_FILE = find_one('NB09A_ROWLEVEL_VS_EGGDISJOINT_COMPARISON.csv', required=False)
NB05_MAP_FILE  = find_one('NB05_wavelength_order_map.csv', required=RUN_PART_C)
NB05B_MAP_FILE = find_one('NB05B_wavelength_order_map.csv', required=RUN_PART_C)
print('Reference inputs located.')

In [ ]:
# ------------------ data, frozen splits, preprocessing (identical to NB11/NB15) ------------------
df = pd.read_csv(DATA_FILE)
outer = pd.read_csv(OUTER_FILE)
spec_cols = sorted([c for c in df.columns if c.startswith('Spectra_')], key=lambda c: float(c.replace('Spectra_', '')))
X_all = df[spec_cols].to_numpy(dtype=np.float32)
y_all = df['storage_days'].to_numpy(dtype=np.float32)
samples = df['sample'].to_numpy()
days = df['storage_days'].to_numpy()
assert X_all.shape == (660, 331)
N_FEATURES = X_all.shape[1]
EGGS = sorted(df['sample'].unique())
assert len(EGGS) == 30 and all((df['sample'] == e).sum() == 22 for e in EGGS)

outer_of_egg = dict(zip(outer['sample'], outer['outer_fold']))
inner_maps = {f: pd.read_csv(SPLIT_DIR / f'inner_group_assignment_outer{f:02d}.csv') for f in range(1, N_OUTER + 1)}
for f in range(1, N_OUTER + 1):
    test_eggs = {e for e, o in outer_of_egg.items() if o == f}
    tr_eggs = set(inner_maps[f]['sample'])
    assert len(test_eggs) == 6 and len(tr_eggs) == 24 and not (test_eggs & tr_eggs)
    assert sorted(inner_maps[f]['inner_fold'].unique()) == list(range(1, N_INNER + 1))
print('Frozen splits OK: 5 outer folds (24 train / 6 test eggs), 4 inner folds each, zero overlap.')

class Prep:
    def __init__(self, name):
        self.name, self.reference_, self.scaler_ = name, None, None
    def _base(self, X):
        X = np.asarray(X, dtype=np.float64)
        if self.name == 'raw': return X.copy()
        if self.name == 'snv':
            mu = X.mean(axis=1, keepdims=True); sd = X.std(axis=1, ddof=1, keepdims=True)
            return (X - mu) / np.where(sd < 1e-12, 1.0, sd)
        if self.name == 'msc':
            ref = self.reference_; rm = ref.mean(); rc = ref - rm; den = np.dot(rc, rc)
            out = np.empty_like(X)
            for i, x in enumerate(X):
                xm = x.mean(); b = np.dot(rc, x - xm) / den
                b = 1.0 if abs(b) < 1e-12 else b
                out[i] = (x - (xm - b * rm)) / b
            return out
        if self.name == 'sg_smooth':
            return savgol_filter(X, SG_WINDOW, SG_POLYORDER, deriv=0, axis=1, mode='interp')
        if self.name == 'sg_deriv1':
            return savgol_filter(X, SG_WINDOW, SG_POLYORDER, deriv=1, delta=1.0, axis=1, mode='interp')
        raise ValueError(self.name)
    def fit(self, X):
        if self.name == 'msc': self.reference_ = np.asarray(X, dtype=np.float64).mean(axis=0)
        self.scaler_ = StandardScaler().fit(self._base(X)); return self
    def transform(self, X):
        return self.scaler_.transform(self._base(X)).astype(np.float32)
    def fit_transform(self, X):
        return self.fit(X).transform(X)

def rows_of(eggs):
    return df['sample'].isin(set(eggs)).to_numpy()

In [ ]:
# ------------------------------ models -----------------------------------
POS_CHANNEL = np.linspace(-1.0, 1.0, N_FEATURES, dtype=np.float32)   # fixed, target-independent

def build_model(variant):
    n_ch = 2 if variant == 'CNN1D_GAP_POS' else 1
    inp = keras.Input(shape=(N_FEATURES, n_ch))
    if variant in ('CNN1D', 'CNN1D_GAP', 'CNN1D_GAP_POS'):
        x = layers.Conv1D(16, 7, padding='same')(inp); x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x); x = layers.MaxPooling1D(2)(x)
        x = layers.Conv1D(32, 5, padding='same')(x); x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x); x = layers.GlobalAveragePooling1D()(x)
    elif variant == 'CNN1D_FLAT':
        x = layers.Conv1D(16, 7, padding='same')(inp); x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x); x = layers.MaxPooling1D(4)(x)
        x = layers.Conv1D(32, 5, padding='same')(x); x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x); x = layers.MaxPooling1D(4)(x)
        x = layers.Flatten()(x)
    else:
        raise ValueError(variant)
    x = layers.Dense(32, activation='relu')(x); x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(1)(x)
    m = keras.Model(inp, out, name=variant)
    m.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss='mse',
              metrics=[keras.metrics.MeanAbsoluteError(name='mae')])
    return m

def param_counts(m):
    tr = int(sum(np.prod(w.shape) for w in m.trainable_weights))
    nt = int(sum(np.prod(w.shape) for w in m.non_trainable_weights))
    return tr, nt, tr + nt

def make_input(variant, X2d):
    X = np.asarray(X2d, dtype=np.float32)[..., None]
    if variant == 'CNN1D_GAP_POS':
        pos = np.broadcast_to(POS_CHANNEL[None, :, None], (X.shape[0], N_FEATURES, 1))
        X = np.concatenate([X, pos], axis=-1)
    return np.ascontiguousarray(X)

def set_seeds(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)

# parameter audit (also documents that the reference architecture matches NB11 / NB15)
audit = []
for v in ['CNN1D', 'CNN1D_GAP_POS', 'CNN1D_FLAT']:
    keras.backend.clear_session()
    tr, nt, tot = param_counts(build_model(v))
    audit.append({'variant': v, 'trainable': tr, 'non_trainable': nt, 'total': tot})
audit = pd.DataFrame(audit)
audit.to_csv(RESULT_DIR / 'NB16_parameter_count_audit.csv', index=False)
display(audit)
assert (audit.loc[audit.variant == 'CNN1D', ['trainable', 'non_trainable', 'total']].iloc[0].tolist() == [3905, 96, 4001]), \
    'Reference CNN1D does not reproduce the NB11/NB15 parameter count.'

def keras_best_epoch(history_val_mae):
    """Best epoch (1-based) under Keras EarlyStopping logic: improvement only if value < best - min_delta."""
    best, best_ep = np.inf, 0
    for i, v in enumerate(history_val_mae):
        if v < best - MIN_DELTA:
            best, best_ep = v, i
    return best_ep + 1

def train_early_stopping(variant, Xtr, ytr, Xva, yva, seed):
    keras.backend.clear_session(); gc.collect(); set_seeds(seed)
    m = build_model(variant)
    es = keras.callbacks.EarlyStopping(monitor='val_mae', mode='min', patience=PATIENCE, min_delta=MIN_DELTA,
                                       restore_best_weights=True, verbose=0)
    t0 = time.perf_counter()
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=MAX_INNER_EPOCHS, batch_size=BATCH_SIZE,
              verbose=0, shuffle=True, callbacks=[es])
    val = h.history['val_mae']
    best_epoch = keras_best_epoch(val)
    kb = getattr(es, 'best_epoch', None)
    if kb is not None and int(kb) + 1 != best_epoch:
        print(f'  note: own best_epoch {best_epoch} vs Keras {int(kb)+1} (using Keras value)')
        best_epoch = int(kb) + 1
    pred = m.predict(Xva, verbose=0).ravel()
    return best_epoch, len(val), time.perf_counter() - t0, pred

def fit_fixed_epochs(variant, Xtr, ytr, epochs, seed):
    keras.backend.clear_session(); gc.collect(); set_seeds(seed)
    m = build_model(variant)
    m.fit(Xtr, ytr, epochs=int(epochs), batch_size=BATCH_SIZE, verbose=0, shuffle=True)
    return m

def pooled_metrics(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float); e = p - y
    return {'MAE_days': float(np.mean(np.abs(e))), 'RMSE_days': float(np.sqrt(np.mean(e ** 2))),
            'R2': float(1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2)), 'bias_days': float(np.mean(e)),
            'median_AE_days': float(np.median(np.abs(e))),
            'within_1d_pct': float(100 * np.mean(np.abs(e) <= 1)), 'within_2d_pct': float(100 * np.mean(np.abs(e) <= 2)),
            'within_3d_pct': float(100 * np.mean(np.abs(e) <= 3))}
print('Model utilities ready.')

In [ ]:
# ------------------------ statistics helpers (same conventions as NB12) ------------------------
def per_egg_mae_from_long(oof_long, model, value_col='y_pred'):
    d = oof_long[oof_long['model'] == model].copy()
    d['ae'] = (d[value_col] - d['storage_days']).abs()
    return d.groupby('sample')['ae'].mean().sort_index()

def egg_arrays(oof_long, models):
    """Return Y (E,22) and P (M,E,22) sorted by egg and storage day."""
    key = df[['sample', 'storage_days']].copy(); key['ord'] = np.arange(len(key))
    key = key.sort_values(['sample', 'storage_days'])
    eggs = sorted(key['sample'].unique())
    Y = key['storage_days'].to_numpy(float).reshape(len(eggs), 22)
    P = []
    for m in models:
        d = oof_long[oof_long['model'] == m][['sample', 'storage_days', 'y_pred']]
        d = key.merge(d, on=['sample', 'storage_days'], how='left').sort_values(['sample', 'storage_days'])
        assert d['y_pred'].notna().all(), f'missing predictions for {m}'
        P.append(d['y_pred'].to_numpy(float).reshape(len(eggs), 22))
    return eggs, Y, np.stack(P)

def cluster_bootstrap(Y, P, models, reps=BOOT_REPS, seed=BOOT_SEED, chunk=500):
    """Whole-egg bootstrap of pooled MAE/RMSE/R2 and of paired MAE differences."""
    M, E, _ = P.shape
    rng = np.random.default_rng(seed)
    idx_all = rng.integers(0, E, size=(reps, E))
    mae = np.empty((reps, M)); rmse = np.empty((reps, M)); r2 = np.empty((reps, M))
    for s in range(0, reps, chunk):
        idx = idx_all[s:s + chunk]
        Yb = Y[idx]                                   # (b,E,22)
        for k in range(M):
            err = P[k][idx] - Yb
            mae[s:s + chunk, k] = np.abs(err).mean(axis=(1, 2))
            rmse[s:s + chunk, k] = np.sqrt((err ** 2).mean(axis=(1, 2)))
            sst = ((Yb - Yb.mean(axis=(1, 2), keepdims=True)) ** 2).sum(axis=(1, 2))
            r2[s:s + chunk, k] = 1 - (err ** 2).sum(axis=(1, 2)) / sst
    per_egg = np.abs(P - Y[None]).mean(axis=2)        # (M,E)
    rows = []
    for k, m in enumerate(models):
        rows.append({'model': m, **{f'{n}_{q}': float(np.percentile(a[:, k], p)) for n, a in
                     [('MAE', mae), ('RMSE', rmse), ('R2', r2)] for q, p in [('CI_low', 2.5), ('CI_high', 97.5)]}})
    return pd.DataFrame(rows).set_index('model'), per_egg, idx_all

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * p[i])); adj[i] = run
    return adj

def rank_biserial(d):
    d = np.asarray(d, float); d = d[d != 0]
    if len(d) == 0: return 0.0
    r = stats.rankdata(np.abs(d)); wp = r[d > 0].sum(); wn = r[d < 0].sum()
    return float((wp - wn) / (wp + wn))

def pairwise_table(per_egg_df, models, idx_all=None, model_pos=None):
    """Two-sided paired Wilcoxon on per-egg MAE, Holm within all pairs, rank-biserial r, optional paired bootstrap CI."""
    rows = []
    for i in range(len(models)):
        for j in range(i + 1, len(models)):
            a, b = models[i], models[j]
            d = per_egg_df[a].to_numpy() - per_egg_df[b].to_numpy()
            p = stats.wilcoxon(per_egg_df[a], per_egg_df[b], alternative='two-sided').pvalue if np.any(d != 0) else 1.0
            row = {'model_A': a, 'model_B': b, 'mean_MAE_A': per_egg_df[a].mean(), 'mean_MAE_B': per_egg_df[b].mean(),
                   'delta_MAE_A_minus_B': d.mean(), 'p_raw': p, 'rank_biserial_r': rank_biserial(d),
                   'n_eggs_A_lower': int((d < 0).sum())}
            if idx_all is not None:
                boot = d[idx_all].mean(axis=1)
                row['delta_CI_low'], row['delta_CI_high'] = np.percentile(boot, [2.5, 97.5])
            rows.append(row)
    t = pd.DataFrame(rows); t['p_holm'] = holm(t['p_raw'].to_numpy()); return t

def friedman_kendall(per_egg_df, models):
    chi2, p = stats.friedmanchisquare(*[per_egg_df[m].to_numpy() for m in models])
    n, k = len(per_egg_df), len(models)
    ranks = per_egg_df[models].rank(axis=1).mean()
    return {'chi2': float(chi2), 'df': k - 1, 'p': float(p), 'kendall_W': float(chi2 / (n * (k - 1))), 'n_eggs': n,
            'k_models': k, 'mean_ranks': ranks.round(2).to_dict()}
print('Statistics helpers ready.')

## Part A — nested benchmark of the position-aware variants (resumable)
For every variant, outer fold, preprocessing candidate and inner fold the network is trained with early stopping
(patience 20, min_delta 0.001, best weights restored, max 250 epochs). The preprocessing with the lowest mean inner MAE
is selected per outer fold; the epoch count is the **median best inner-fold epoch** of that preprocessing (NB11 rule).
The final model is refitted on the 24 outer-training eggs for that fixed number of epochs with seeds 2026, 2027 and 2028
and evaluated once on the 6 unseen outer-test eggs.

In [6]:
if RUN_PART_A:
    INNER_LOG = CKPT_DIR / 'NB16_inner_foldwise.csv'
    done = set()
    if INNER_LOG.exists():
        prev = pd.read_csv(INNER_LOG)
        done = set(zip(prev['variant'], prev['outer_fold'], prev['preprocessing'], prev['inner_fold']))
        print(f'Resuming: {len(done)} inner fits already completed.')
    n_total = len(NEW_VARIANTS) * len(OUTER_FOLDS) * len(PREPS) * N_INNER
    n_done = len(done); t_start = time.time()
    for variant in NEW_VARIANTS:
        for of in OUTER_FOLDS:
            inner = inner_maps[of]
            for prep in PREPS:
                p_idx = PREP_ORDER.index(prep)
                for k in range(1, N_INNER + 1):
                    key = (variant, of, prep, k)
                    if key in done: continue
                    va_eggs = inner.loc[inner['inner_fold'] == k, 'sample'].tolist()
                    tr_eggs = inner.loc[inner['inner_fold'] != k, 'sample'].tolist()
                    assert len(va_eggs) == 6 and len(tr_eggs) == 18
                    tr, va = rows_of(tr_eggs), rows_of(va_eggs)
                    pp = Prep(prep); Xtr = pp.fit_transform(X_all[tr]); Xva = pp.transform(X_all[va])
                    seed = SEED_BASE[variant] + 100 * of + 10 * p_idx + k
                    be, ran, secs, pred = train_early_stopping(variant, make_input(variant, Xtr), y_all[tr],
                                                               make_input(variant, Xva), y_all[va], seed)
                    row = {'variant': variant, 'outer_fold': of, 'inner_fold': k, 'preprocessing': prep, 'seed': seed,
                           'best_epoch': be, 'epochs_ran': ran, 'fit_seconds': round(secs, 2),
                           'train_eggs': len(tr_eggs), 'val_eggs': len(va_eggs), **pooled_metrics(y_all[va], pred)}
                    pd.DataFrame([row]).to_csv(INNER_LOG, mode='a', header=not INNER_LOG.exists(), index=False)
                    done.add(key); n_done += 1
                    el = time.time() - t_start
                    print(f'[{n_done}/{n_total}] {variant} outer {of} {prep:<9} inner {k} | best_epoch {be:>3} | '
                          f'val MAE {row["MAE_days"]:.3f} | {secs:5.1f}s | elapsed {el/60:5.1f} min', flush=True)
    inner_df = pd.read_csv(INNER_LOG)
    inner_df.to_csv(RESULT_DIR / 'NB16_inner_foldwise.csv', index=False)

    # selection (NB11 rule)
    sel = []
    for (variant, of), g in inner_df.groupby(['variant', 'outer_fold']):
        agg = g.groupby('preprocessing')['MAE_days'].agg(['mean', 'std']).reset_index().sort_values('mean')
        best = agg.iloc[0]
        be = g.loc[g['preprocessing'] == best['preprocessing'], 'best_epoch'].to_numpy()
        sel.append({'outer_fold': of, 'variant': variant, 'preprocessing': best['preprocessing'],
                    'selected_epoch': int(np.floor(np.median(be) + 0.5)), 'mean_inner_MAE_days': float(best['mean']),
                    'sd_inner_MAE_days': float(best['std']),
                    'selection_rule': 'minimum mean MAE across four frozen inner egg-disjoint folds; epochs = median best inner epoch'})
    sel = pd.DataFrame(sel).sort_values(['variant', 'outer_fold']).reset_index(drop=True)
    sel.to_csv(RESULT_DIR / 'NB16_selected_configurations.csv', index=False)
    display(sel)

[1/200] CNN1D_FLAT outer 1 raw       inner 1 | best_epoch  86 | val MAE 3.255 |  27.5s | elapsed   0.5 min
[2/200] CNN1D_FLAT outer 1 raw       inner 2 | best_epoch  35 | val MAE 4.576 |  15.9s | elapsed   0.7 min


[3/200] CNN1D_FLAT outer 1 raw       inner 3 | best_epoch 110 | val MAE 3.145 |  26.1s | elapsed   1.2 min
[4/200] CNN1D_FLAT outer 1 raw       inner 4 | best_epoch  87 | val MAE 3.271 |  22.9s | elapsed   1.6 min
[5/200] CNN1D_FLAT outer 1 snv       inner 1 | best_epoch  72 | val MAE 3.113 |  20.8s | elapsed   1.9 min
[6/200] CNN1D_FLAT outer 1 snv       inner 2 | best_epoch  74 | val MAE 3.466 |  21.3s | elapsed   2.3 min
[7/200] CNN1D_FLAT outer 1 snv       inner 3 | best_epoch  64 | val MAE 2.789 |  19.9s | elapsed   2.6 min
[8/200] CNN1D_FLAT outer 1 snv       inner 4 | best_epoch  68 | val MAE 2.810 |  20.4s | elapsed   3.0 min
[9/200] CNN1D_FLAT outer 1 msc       inner 1 | best_epoch  80 | val MAE 2.704 |  22.0s | elapsed   3.4 min
[10/200] CNN1D_FLAT outer 1 msc       inner 2 | best_epoch  48 | val MAE 3.519 |  17.7s | elapsed   3.7 min
[11/200] CNN1D_FLAT outer 1 msc       inner 3 | best_epoch  50 | val MAE 2.750 |  17.8s | elapsed   4.0 min
[12/200] CNN1D_FLAT outer 1 msc    

,outer_fold,variant,preprocessing,selected_epoch,mean_inner_MAE_days,sd_inner_MAE_days,selection_rule
0,1,CNN1D_FLAT,sg_deriv1,44,2.710672,0.243800,minimum mean MAE across four frozen inner egg-...
1,2,CNN1D_FLAT,sg_deriv1,55,2.446556,0.249608,minimum mean MAE across four frozen inner egg-...
2,3,CNN1D_FLAT,sg_deriv1,60,2.319714,0.253347,minimum mean MAE across four frozen inner egg-...
3,4,CNN1D_FLAT,sg_deriv1,51,2.477928,0.215574,minimum mean MAE across four frozen inner egg-...
4,5,CNN1D_FLAT,sg_deriv1,74,2.487587,0.259765,minimum mean MAE across four frozen inner egg-...
5,1,CNN1D_GAP_POS,sg_deriv1,60,4.228877,0.581702,minimum mean MAE across four frozen inner egg-...
6,2,CNN1D_GAP_POS,sg_deriv1,46,4.271400,0.205210,minimum mean MAE across four frozen inner egg-...
7,3,CNN1D_GAP_POS,raw,92,4.507834,0.232978,minimum mean MAE across four frozen inner egg-...
8,4,CNN1D_GAP_POS,sg_deriv1,42,4.496502,0.259007,minimum mean MAE across four frozen inner egg-...
9,5,CNN1D_GAP_POS,sg_deriv1,40,4.433095,0.357572,minimum mean MAE across four frozen inner egg-...


In [ ]:
if RUN_PART_A:
    sel = pd.read_csv(RESULT_DIR / 'NB16_selected_configurations.csv')
    sw_parts = []; n_total = len(NEW_VARIANTS) * len(OUTER_FOLDS) * len(FINAL_SEEDS); n_done = 0; t_start = time.time()
    for variant in NEW_VARIANTS:
        for of in OUTER_FOLDS:
            r = sel[(sel['variant'] == variant) & (sel['outer_fold'] == of)].iloc[0]
            prep, epochs = str(r['preprocessing']), int(r['selected_epoch'])
            te_eggs = sorted(e for e, o in outer_of_egg.items() if o == of)
            tr_eggs = sorted(inner_maps[of]['sample'])
            tr, te = rows_of(tr_eggs), rows_of(te_eggs)
            assert not (set(tr_eggs) & set(te_eggs)) and tr.sum() == 528 and te.sum() == 132
            pp = Prep(prep); Xtr = pp.fit_transform(X_all[tr]); Xte = pp.transform(X_all[te])
            for seed in FINAL_SEEDS:
                ck = CKPT_DIR / f'final_{variant}_o{of}_s{seed}.csv'
                if not ck.exists():
                    m = fit_fixed_epochs(variant, make_input(variant, Xtr), y_all[tr], epochs, seed)
                    pred = m.predict(make_input(variant, Xte), verbose=0).ravel()
                    pd.DataFrame({'sample': samples[te], 'storage_days': days[te], 'outer_fold': of, 'model': variant,
                                  'seed': seed, 'y_pred': pred}).to_csv(ck, index=False)
                n_done += 1
                print(f'[{n_done}/{n_total}] final {variant} outer {of} seed {seed} | prep {prep} | epochs {epochs} | '
                      f'elapsed {(time.time()-t_start)/60:5.1f} min', flush=True)
                sw_parts.append(pd.read_csv(ck))
    oof_sw = pd.concat(sw_parts, ignore_index=True)
    oof_sm = (oof_sw.groupby(['sample', 'storage_days', 'outer_fold', 'model'], as_index=False)
              .agg(y_pred=('y_pred', 'mean'), seed_sd=('y_pred', 'std')))
    for v in NEW_VARIANTS:
        assert (oof_sm['model'] == v).sum() == 660, f'{v}: expected 660 out-of-fold rows'
    oof_sw.to_csv(RESULT_DIR / 'NB16_oof_predictions_seedwise.csv', index=False)
    oof_sm.to_csv(RESULT_DIR / 'NB16_oof_predictions_seedmean.csv', index=False)
    print('OOF predictions written:', oof_sm.shape)

[1/30] final CNN1D_FLAT outer 1 seed 2026 | prep sg_deriv1 | epochs 44 | elapsed   0.3 min
[2/30] final CNN1D_FLAT outer 1 seed 2027 | prep sg_deriv1 | epochs 44 | elapsed   0.5 min
[3/30] final CNN1D_FLAT outer 1 seed 2028 | prep sg_deriv1 | epochs 44 | elapsed   0.8 min
[4/30] final CNN1D_FLAT outer 2 seed 2026 | prep sg_deriv1 | epochs 55 | elapsed   1.1 min
[5/30] final CNN1D_FLAT outer 2 seed 2027 | prep sg_deriv1 | epochs 55 | elapsed   1.3 min
[6/30] final CNN1D_FLAT outer 2 seed 2028 | prep sg_deriv1 | epochs 55 | elapsed   1.6 min
[7/30] final CNN1D_FLAT outer 3 seed 2026 | prep sg_deriv1 | epochs 60 | elapsed   1.9 min
[8/30] final CNN1D_FLAT outer 3 seed 2027 | prep sg_deriv1 | epochs 60 | elapsed   2.2 min
[9/30] final CNN1D_FLAT outer 3 seed 2028 | prep sg_deriv1 | epochs 60 | elapsed   2.5 min
[10/30] final CNN1D_FLAT outer 4 seed 2026 | prep sg_deriv1 | epochs 51 | elapsed   2.8 min
[11/30] final CNN1D_FLAT outer 4 seed 2027 | prep sg_deriv1 | epochs 51 | elapsed   3.0 m

In [ ]:
if RUN_PART_A:
    oof_sm = pd.read_csv(RESULT_DIR / 'NB16_oof_predictions_seedmean.csv')
    oof_sw = pd.read_csv(RESULT_DIR / 'NB16_oof_predictions_seedwise.csv')

    # reference OOF (frozen seven-model unified file from NB12); validate ingestion against the frozen per-egg table
    ref_oof = pd.read_csv(NB12_UNIFIED_FILE)
    ref_wide = pd.read_csv(NB12_PEREGG_FILE).set_index('sample').sort_index()
    for m in ['SVR', 'PLSR', 'ANN', REF]:
        chk = per_egg_mae_from_long(ref_oof, m)
        assert np.allclose(chk.values, ref_wide[m].values, atol=1e-6), f'Ingestion check failed for {m}'
    print('PASS - reference OOF reproduces the frozen NB12 per-egg MAE table.')

    MODELS = ['SVR', 'PLSR', 'ANN', REF] + NEW_VARIANTS
    both = pd.concat([ref_oof[ref_oof['model'].isin(['SVR', 'PLSR', 'ANN', REF])][['sample', 'storage_days', 'model', 'y_pred']],
                      oof_sm[['sample', 'storage_days', 'model', 'y_pred']]], ignore_index=True)
    eggs, Y, P = egg_arrays(both, MODELS)
    ci, per_egg, idx_all = cluster_bootstrap(Y, P, MODELS)

    # pooled metrics (seed-mean predictions), with whole-egg bootstrap CIs
    rows = []
    for k, m in enumerate(MODELS):
        rows.append({'model': m, **pooled_metrics(Y.ravel(), P[k].ravel()), **ci.loc[m].to_dict()})
    perf = pd.DataFrame(rows)
    perf.to_csv(RESULT_DIR / 'Table_NB16_performance_CNN_variants.csv', index=False)
    display(perf.round(3))

    per_egg_df = pd.DataFrame(per_egg.T, columns=MODELS, index=eggs); per_egg_df.index.name = 'sample'
    per_egg_df.to_csv(RESULT_DIR / 'NB16_per_egg_MAE_wide.csv')

    fk = friedman_kendall(per_egg_df, MODELS)
    pw = pairwise_table(per_egg_df, MODELS, idx_all=idx_all)
    pw.to_csv(RESULT_DIR / 'NB16_pairwise_wilcoxon_holm.csv', index=False)
    (RESULT_DIR / 'NB16_friedman_kendall.json').write_text(json.dumps(fk, indent=2), encoding='utf-8')
    print('Friedman:', {k: fk[k] for k in ['chi2', 'df', 'p', 'kendall_W']})
    display(pw[pw['model_A'].isin(NEW_VARIANTS) | pw['model_B'].isin(NEW_VARIANTS)].round(4))

    # seed-wise (per-seed) pooled metrics and seed stability
    sw_rows = []
    for (v, sd), g in oof_sw.groupby(['model', 'seed']):
        sw_rows.append({'model': v, 'seed': sd, **pooled_metrics(g['storage_days'].to_numpy(), g['y_pred'].to_numpy())})
    pd.DataFrame(sw_rows).to_csv(RESULT_DIR / 'NB16_pooled_seedwise_metrics.csv', index=False)

    # by-day metrics for the new variants (bias / MAE), as in NB07/NB11
    byday = []
    for v in NEW_VARIANTS:
        d = oof_sm[oof_sm['model'] == v].copy(); d['err'] = d['y_pred'] - d['storage_days']
        g = d.groupby('storage_days')['err']
        byday.append(pd.DataFrame({'model': v, 'storage_days': g.mean().index, 'bias_days': g.mean().values,
                                   'MAE_days': d.assign(ae=d['err'].abs()).groupby('storage_days')['ae'].mean().values}))
    pd.concat(byday).to_csv(RESULT_DIR / 'NB16_metrics_by_storage_day.csv', index=False)

    # parameter-matched sentence anchors
    def f(x, n=3): return f'{x:.{n}f}'
    a = []
    for v in NEW_VARIANTS:
        r = perf[perf['model'] == v].iloc[0]
        a.append(f'{v}: MAE = {f(r.MAE_days)} days (egg-cluster bootstrap 95% CI {f(r.MAE_CI_low)}-{f(r.MAE_CI_high)}), '
                 f'RMSE = {f(r.RMSE_days)} days, R2 = {f(r.R2)}.')
    r0 = perf[perf['model'] == REF].iloc[0]
    a.append(f'Reference CNN1D (GAP): MAE = {f(r0.MAE_days)} days, R2 = {f(r0.R2)}.')
    a.append(f'Friedman across {len(MODELS)} models: chi2({fk["df"]}) = {f(fk["chi2"])}, p = {fk["p"]:.3g}, Kendall W = {f(fk["kendall_W"])}.')
    (RESULT_DIR / 'NB16_partA_wording_anchors.md').write_text('# Part A wording anchors (generated from saved CSV files)\n\n' +
                                                              '\n\n'.join(f'{i+1}. {s}' for i, s in enumerate(a)), encoding='utf-8')
    print('\n'.join(a))

PASS - reference OOF reproduces the frozen NB12 per-egg MAE table.


,model,MAE_days,RMSE_days,R2,bias_days,median_AE_days,within_1d_pct,within_2d_pct,within_3d_pct,MAE_CI_low,MAE_CI_high,RMSE_CI_low,RMSE_CI_high,R2_CI_low,R2_CI_high
0,SVR,2.195,2.716,0.817,0.148,1.955,27.727,50.909,73.030,2.027,2.371,2.527,2.912,0.789,0.841
1,PLSR,2.267,2.864,0.796,0.006,1.870,25.909,52.879,70.909,2.084,2.490,2.619,3.142,0.755,0.830
2,ANN,2.289,2.974,0.780,-0.159,1.836,30.455,53.939,70.303,2.096,2.519,2.723,3.255,0.737,0.816
3,CNN1D,4.572,5.623,0.214,0.022,4.002,12.727,25.152,40.152,4.092,5.088,5.060,6.198,0.046,0.364
4,CNN1D_FLAT,2.469,3.195,0.746,-0.359,1.998,27.576,50.000,68.182,2.241,2.767,2.896,3.567,0.684,0.792
5,CNN1D_GAP_POS,4.766,5.786,0.168,0.695,4.247,11.212,23.788,34.394,4.388,5.146,5.321,6.244,0.031,0.297


Friedman: {'chi2': 104.24761904761908, 'df': 5, 'p': 6.718496849623474e-21, 'kendall_W': 0.6949841269841273}


,model_A,model_B,mean_MAE_A,mean_MAE_B,delta_MAE_A_minus_B,p_raw,rank_biserial_r,n_eggs_A_lower,delta_CI_low,delta_CI_high,p_holm
3,SVR,CNN1D_FLAT,2.1946,2.4689,-0.2742,0.0384,-0.4323,19,-0.5403,-0.0430,0.2305
4,SVR,CNN1D_GAP_POS,2.1946,4.7664,-2.5718,0.0000,-1.0000,30,-2.9481,-2.1818,0.0000
7,PLSR,CNN1D_FLAT,2.2673,2.4689,-0.2016,0.0175,-0.4925,21,-0.3700,-0.0351,0.1222
8,PLSR,CNN1D_GAP_POS,2.2673,4.7664,-2.4991,0.0000,-1.0000,30,-2.8884,-2.0908,0.0000
10,ANN,CNN1D_FLAT,2.2889,2.4689,-0.1799,0.1347,-0.3161,18,-0.3987,0.0356,0.6468
11,ANN,CNN1D_GAP_POS,2.2889,4.7664,-2.4774,0.0000,-1.0000,30,-2.8909,-2.0735,0.0000
12,CNN1D,CNN1D_FLAT,4.5719,2.4689,2.1031,0.0000,1.0000,0,1.6966,2.5177,0.0000
13,CNN1D,CNN1D_GAP_POS,4.5719,4.7664,-0.1945,0.1294,-0.3204,18,-0.4932,0.1183,0.6468
14,CNN1D_FLAT,CNN1D_GAP_POS,2.4689,4.7664,-2.2975,0.0000,-0.9914,29,-2.6804,-1.9039,0.0000


CNN1D_FLAT: MAE = 2.469 days (egg-cluster bootstrap 95% CI 2.241-2.767), RMSE = 3.195 days, R2 = 0.746.
CNN1D_GAP_POS: MAE = 4.766 days (egg-cluster bootstrap 95% CI 4.388-5.146), RMSE = 5.786 days, R2 = 0.168.
Reference CNN1D (GAP): MAE = 4.572 days, R2 = 0.214.
Friedman across 6 models: chi2(5) = 104.248, p = 6.72e-21, Kendall W = 0.695.


## Part B — row-level (leaky) diagnostic for the CNN variants (extends Table 4)
Uses the **same five random row-level folds as NB09A** (read from `NB09A_rowlevel_oof_seedmean.csv`), the same three seeds
and the NB09A convention for "representative" configurations: the modal selected preprocessing and the median selected epoch
across the five egg-disjoint outer folds. Every egg appears in both training and test partitions by design.

In [ ]:
if RUN_PART_B:
    rl = pd.read_csv(NB09A_ROWLEVEL_FILE)
    dm = rl[rl['model'] == 'DummyMean'][['row_index', 'sample', 'storage_days', 'row_fold']].drop_duplicates('row_index')
    assert len(dm) == 660 and dm['row_fold'].nunique() == 5, 'unexpected NB09A row-level fold file'
    offset = None
    for off in (0, -1, 1):
        idx = dm['row_index'].to_numpy() + off
        if idx.min() >= 0 and idx.max() < len(df) and (samples[idx] == dm['sample'].to_numpy()).all() \
                and np.allclose(days[idx], dm['storage_days'].to_numpy()):
            offset = off; break
    assert offset is not None, 'Could not align NB09A row_index with the dataset rows.'
    row_fold = np.zeros(len(df), dtype=int); row_fold[dm['row_index'].to_numpy() + offset] = dm['row_fold'].to_numpy()
    assert set(row_fold) == {1, 2, 3, 4, 5}
    print('NB09A row-level folds aligned (row_index offset =', offset, ').')

    # representative configurations (NB09A convention)
    nb11_sel = pd.read_csv(NB11_SELECTED_FILE)
    rep = {'CNN1D': (nb11_sel['preprocessing'].mode().iloc[0], int(np.floor(nb11_sel['selected_epoch'].median() + 0.5)))}
    if RUN_PART_A or (RESULT_DIR / 'NB16_selected_configurations.csv').exists():
        s16 = pd.read_csv(RESULT_DIR / 'NB16_selected_configurations.csv')
        for v in NEW_VARIANTS:
            g = s16[s16['variant'] == v]
            rep[v] = (g['preprocessing'].mode().iloc[0], int(np.floor(g['selected_epoch'].median() + 0.5)))
    print('Representative configurations:', rep)
    (RESULT_DIR / 'NB16_rowlevel_representative_configurations.json').write_text(json.dumps(
        {k: {'preprocessing': v[0], 'epochs': v[1]} for k, v in rep.items()}, indent=2), encoding='utf-8')

    part = []
    build_name = {'CNN1D': 'CNN1D'}   # 'CNN1D' is the GAP reference architecture in build_model
    for variant, (prep, epochs) in rep.items():
        for fold in range(1, 6):
            for seed in FINAL_SEEDS:
                ck = CKPT_DIR / f'rowlevel_{variant}_f{fold}_s{seed}.csv'
                if not ck.exists():
                    tr, te = row_fold != fold, row_fold == fold
                    pp = Prep(prep); Xtr = pp.fit_transform(X_all[tr]); Xte = pp.transform(X_all[te])
                    m = fit_fixed_epochs(variant, make_input(variant, Xtr), y_all[tr], epochs, seed)
                    pd.DataFrame({'row_index': np.where(te)[0], 'sample': samples[te], 'storage_days': days[te],
                                  'row_fold': fold, 'model': variant, 'seed': seed,
                                  'y_pred': m.predict(make_input(variant, Xte), verbose=0).ravel()}).to_csv(ck, index=False)
                print(f'row-level {variant} fold {fold} seed {seed} done', flush=True)
                part.append(pd.read_csv(ck))
    rl_sw = pd.concat(part, ignore_index=True)
    rl_sm = (rl_sw.groupby(['row_index', 'sample', 'storage_days', 'row_fold', 'model'], as_index=False)
             .agg(y_pred=('y_pred', 'mean'), seed_sd=('y_pred', 'std')))
    rl_sw.to_csv(RESULT_DIR / 'NB16_rowlevel_oof_seedwise.csv', index=False)
    rl_sm.to_csv(RESULT_DIR / 'NB16_rowlevel_oof_seedmean.csv', index=False)

    # egg-disjoint reference metrics for the same models
    ref_oof = pd.read_csv(NB12_UNIFIED_FILE)
    egg_disjoint = {}
    ref_ = ref_oof[ref_oof['model'] == 'CNN1D']
    egg_disjoint['CNN1D'] = pooled_metrics(ref_['storage_days'], ref_['y_pred'])
    if RUN_PART_A or (RESULT_DIR / 'NB16_oof_predictions_seedmean.csv').exists():
        o16 = pd.read_csv(RESULT_DIR / 'NB16_oof_predictions_seedmean.csv')
        for v in NEW_VARIANTS:
            g = o16[o16['model'] == v]; egg_disjoint[v] = pooled_metrics(g['storage_days'], g['y_pred'])
    rows = []
    for v in rep:
        g = rl_sm[rl_sm['model'] == v]; rlm = pooled_metrics(g['storage_days'], g['y_pred']); ed = egg_disjoint[v]
        rows.append({'model': v, 'MAE_eggdisjoint': ed['MAE_days'], 'MAE_rowlevel': rlm['MAE_days'],
                     'RMSE_eggdisjoint': ed['RMSE_days'], 'RMSE_rowlevel': rlm['RMSE_days'],
                     'R2_eggdisjoint': ed['R2'], 'R2_rowlevel': rlm['R2'],
                     'MAE_optimism_gap_days': ed['MAE_days'] - rlm['MAE_days'],
                     'MAE_relative_reduction_pct': 100 * (ed['MAE_days'] - rlm['MAE_days']) / ed['MAE_days'],
                     'R2_inflation': rlm['R2'] - ed['R2']})
    t4 = pd.DataFrame(rows); t4.to_csv(RESULT_DIR / 'Table_NB16_rowlevel_vs_eggdisjoint_CNN.csv', index=False)
    display(t4.round(3))
    if NB09A_COMPARISON_FILE is not None:
        old = pd.read_csv(NB09A_COMPARISON_FILE)
        keep = [c for c in ['model', 'MAE_days_eggdisjoint', 'MAE_days_rowlevel', 'RMSE_days_eggdisjoint', 'RMSE_days_rowlevel',
                            'R2_eggdisjoint', 'R2_rowlevel', 'MAE_relative_reduction_pct'] if c in old.columns]
        pd.concat([old[keep], t4.rename(columns={'MAE_eggdisjoint': 'MAE_days_eggdisjoint', 'MAE_rowlevel': 'MAE_days_rowlevel',
                                                 'RMSE_eggdisjoint': 'RMSE_days_eggdisjoint', 'RMSE_rowlevel': 'RMSE_days_rowlevel'})[keep]],
                  ignore_index=True).to_csv(RESULT_DIR / 'Table_4_extended_all_models.csv', index=False)

NB09A row-level folds aligned (row_index offset = 0 ).
Representative configurations: {'CNN1D': ('sg_deriv1', 68), 'CNN1D_FLAT': ('sg_deriv1', 55), 'CNN1D_GAP_POS': ('sg_deriv1', 46)}
row-level CNN1D fold 1 seed 2026 done
row-level CNN1D fold 1 seed 2027 done
row-level CNN1D fold 1 seed 2028 done
row-level CNN1D fold 2 seed 2026 done
row-level CNN1D fold 2 seed 2027 done
row-level CNN1D fold 2 seed 2028 done
row-level CNN1D fold 3 seed 2026 done
row-level CNN1D fold 3 seed 2027 done
row-level CNN1D fold 3 seed 2028 done
row-level CNN1D fold 4 seed 2026 done
row-level CNN1D fold 4 seed 2027 done
row-level CNN1D fold 4 seed 2028 done
row-level CNN1D fold 5 seed 2026 done
row-level CNN1D fold 5 seed 2027 done
row-level CNN1D fold 5 seed 2028 done
row-level CNN1D_FLAT fold 1 seed 2026 done
row-level CNN1D_FLAT fold 1 seed 2027 done
row-level CNN1D_FLAT fold 1 seed 2028 done
row-level CNN1D_FLAT fold 2 seed 2026 done
row-level CNN1D_FLAT fold 2 seed 2027 done
row-level CNN1D_FLAT fold 2 see

,model,MAE_eggdisjoint,MAE_rowlevel,RMSE_eggdisjoint,RMSE_rowlevel,R2_eggdisjoint,R2_rowlevel,MAE_optimism_gap_days,MAE_relative_reduction_pct,R2_inflation
0,CNN1D,4.572,4.412,5.623,5.341,0.214,0.291,0.160,3.505,0.077
1,CNN1D_FLAT,2.469,2.179,3.195,2.828,0.746,0.801,0.290,11.737,0.055
2,CNN1D_GAP_POS,4.766,4.299,5.786,5.196,0.168,0.329,0.468,9.810,0.161


## Part C — wavelength-order ablation for the CNN variants (extends Table 6)
Same recipe as NB05/NB05B: the frozen preprocessing and epoch count of each outer fold are inherited (no retuning); scaling is
fitted in the physical wavelength order and the transformed matrix is then reordered. Permutations are read from the NB05 and
NB05B maps, so the reversed and shuffled spectra are **exactly** those used for ANN/SimpleRNN/LSTM/BiLSTM. The "original" condition
is the frozen OOF of each model (NB11 for `CNN1D`, Part A for `CNN1D_FLAT`). Inference follows NB06: seed-wise per-egg MAE averaged over
seeds, Friedman + Kendall W across the three conditions, Wilcoxon-Holm within model.

In [ ]:
if RUN_PART_C:
    map05 = pd.read_csv(NB05_MAP_FILE); map05b = pd.read_csv(NB05B_MAP_FILE)
    def perm_from(m, col, key):
        g = m[m[col] == key].sort_values('new_position_0based')
        p = g['source_index_0based'].to_numpy(dtype=int)
        assert len(p) == N_FEATURES and sorted(p.tolist()) == list(range(N_FEATURES)), f'invalid permutation for {key}'
        return p
    ORDERS = {'reversed': perm_from(map05, 'condition', 'reversed'), 'shuffled_52026': perm_from(map05, 'condition', 'shuffled')}
    for s in [11017, 24601, 73819, 90210]:
        ORDERS[f'shuffled_{s}'] = perm_from(map05b, 'shuffle_seed', s)
    assert (ORDERS['reversed'] == np.arange(N_FEATURES)[::-1]).all(), 'NB05 reversed map is not a plain reversal'
    print('Permutation maps loaded:', list(ORDERS))

    ABL_MODELS = {'CNN1D': None, 'CNN1D_FLAT': None}
    nb11_sel = pd.read_csv(NB11_SELECTED_FILE)
    s16 = pd.read_csv(RESULT_DIR / 'NB16_selected_configurations.csv')
    def frozen_cfg(model, of):
        if model == 'CNN1D':
            r = nb11_sel[nb11_sel['outer_fold'] == of].iloc[0]
        else:
            r = s16[(s16['variant'] == model) & (s16['outer_fold'] == of)].iloc[0]
        return str(r['preprocessing']), int(r['selected_epoch'])

    abl_parts = []; n_total = len(ABL_MODELS) * len(ORDERS) * len(OUTER_FOLDS) * len(FINAL_SEEDS); n_done = 0; t0 = time.time()
    for model in ABL_MODELS:
        for cond, perm in ORDERS.items():
            for of in OUTER_FOLDS:
                prep, epochs = frozen_cfg(model, of)
                te_eggs = sorted(e for e, o in outer_of_egg.items() if o == of); tr_eggs = sorted(inner_maps[of]['sample'])
                tr, te = rows_of(tr_eggs), rows_of(te_eggs)
                pp = Prep(prep); Xtr = pp.fit_transform(X_all[tr])[:, perm]; Xte = pp.transform(X_all[te])[:, perm]   # scale in physical order, then reorder
                for seed in FINAL_SEEDS:
                    ck = CKPT_DIR / f'ablation_{model}_{cond}_o{of}_s{seed}.csv'
                    if not ck.exists():
                        m = fit_fixed_epochs(model, make_input(model, Xtr), y_all[tr], epochs, seed)
                        pd.DataFrame({'sample': samples[te], 'storage_days': days[te], 'outer_fold': of, 'model': model,
                                      'condition': cond, 'seed': seed,
                                      'y_pred': m.predict(make_input(model, Xte), verbose=0).ravel()}).to_csv(ck, index=False)
                    n_done += 1
                    if n_done % 5 == 0 or n_done == n_total:
                        print(f'[{n_done}/{n_total}] {model} {cond} outer {of} seed {seed} | elapsed {(time.time()-t0)/60:5.1f} min', flush=True)
                    abl_parts.append(pd.read_csv(ck))
    abl_sw = pd.concat(abl_parts, ignore_index=True)
    # original condition = frozen OOF
    orig_parts = [pd.read_csv(NB11_SEEDWISE_FILE).assign(condition='original')[['sample', 'storage_days', 'outer_fold', 'model', 'condition', 'seed', 'y_pred']]]
    orig_parts[0] = orig_parts[0][orig_parts[0]['model'] == 'CNN1D']
    orig_parts.append(pd.read_csv(RESULT_DIR / 'NB16_oof_predictions_seedwise.csv').query("model == 'CNN1D_FLAT'")
                      .assign(condition='original')[['sample', 'storage_days', 'outer_fold', 'model', 'condition', 'seed', 'y_pred']])
    allc = pd.concat(orig_parts + [abl_sw], ignore_index=True)
    allc.to_csv(RESULT_DIR / 'NB16_ablation_oof_seedwise_all_conditions.csv', index=False)

    # per-egg seed-wise MAE (mean over seeds of per-egg MAE), NB06 convention
    allc['ae'] = (allc['y_pred'] - allc['storage_days']).abs()
    pe_seed = allc.groupby(['model', 'condition', 'seed', 'sample'])['ae'].mean().reset_index()
    pe = pe_seed.groupby(['model', 'condition', 'sample'])['ae'].mean().reset_index()
    pe.to_csv(RESULT_DIR / 'NB16_ablation_per_egg_seedwise_MAE.csv', index=False)

    main_conds = ['original', 'reversed', 'shuffled_52026']
    rows, pair_rows = [], []
    for model in ABL_MODELS:
        w = pe[pe['model'] == model].pivot(index='sample', columns='condition', values='ae')[main_conds]
        fk = friedman_kendall(w, main_conds)
        rows.append({'model': model, 'Original_MAE': w['original'].mean(), 'Reversed_MAE': w['reversed'].mean(),
                     'Shuffled_MAE': w['shuffled_52026'].mean(), 'Friedman_p': fk['p'], 'Kendall_W': fk['kendall_W'],
                     'Friedman_chi2': fk['chi2']})
        pt = pairwise_table(w, main_conds); pt.insert(0, 'model', model); pair_rows.append(pt)
    t6 = pd.DataFrame(rows); t6.to_csv(RESULT_DIR / 'Table_NB16_wavelength_order_CNN.csv', index=False)
    pd.concat(pair_rows).to_csv(RESULT_DIR / 'NB16_ablation_pairwise_wilcoxon_holm.csv', index=False)
    display(t6.round(4)); display(pd.concat(pair_rows).round(4))

    # multi-permutation summary (5 shuffles)
    sh = [c for c in ORDERS if c.startswith('shuffled_')]
    msum = []
    for model in ABL_MODELS:
        w = pe[pe['model'] == model].pivot(index='sample', columns='condition', values='ae')
        orig = w['original'].mean()
        vals = {c: w[c].mean() for c in sh}
        msum.append({'model': model, 'original_MAE': orig, 'reversed_MAE': w['reversed'].mean(),
                     **{f'MAE_{c}': v for c, v in vals.items()}, 'shuffled_min': min(vals.values()), 'shuffled_max': max(vals.values()),
                     'n_shuffles_worse_than_original': int(sum(v > orig for v in vals.values())), 'n_shuffles': len(vals)})
    pd.DataFrame(msum).to_csv(RESULT_DIR / 'Table_NB16_multi_permutation_summary.csv', index=False)
    display(pd.DataFrame(msum).round(3))

In [ ]:
# ------------------------- protocol, summary and result package -------------------------
protocol = {
    'notebook': 'NB16_CNN1D_POSITION_AWARE_SENSITIVITY', 'run_revision': RUN_REVISION, 'quick_test': QUICK_TEST,
    'purpose': 'Post hoc test of whether the negative CNN1D result is due to global-average-pooling position invariance',
    'dataset_sha256': dataset_sha, 'frozen_split_manifest_sha256': split_sha, 'strict_integrity': STRICT_INTEGRITY,
    'variants': {'CNN1D': 'reference (GAP), frozen NB11 outputs; re-fitted only in Part C ablation',
                 'CNN1D_GAP_POS': 'GAP architecture + fixed position channel linspace(-1,1)',
                 'CNN1D_FLAT': 'Conv16k7-BN-MP4-Conv32k5-BN-MP4-Flatten-Dense32-Dropout0.2-Linear'},
    'preprocessing_candidates': PREPS, 'outer_folds': N_OUTER, 'inner_folds': N_INNER,
    'optimizer': 'Adam', 'learning_rate': LEARNING_RATE, 'batch_size': BATCH_SIZE, 'max_inner_epochs': MAX_INNER_EPOCHS,
    'early_stopping': {'monitor': 'val_mae', 'patience': PATIENCE, 'min_delta': MIN_DELTA, 'restore_best_weights': True},
    'epoch_rule': 'median best inner-fold epoch for selected preprocessing (rounded half up)',
    'inner_seed_rule': 'base + 100*outer + 10*prep_index + inner; base 62000 (FLAT) / 63000 (GAP_POS)',
    'final_seeds': FINAL_SEEDS, 'bootstrap_replicates': BOOT_REPS, 'bootstrap_seed': BOOT_SEED,
    'bootstrap_scheme': 'resample eggs with replacement; retain all 22 rows within each sampled egg',
    'part_B_row_level_source': 'row_fold column of NB09A_rowlevel_oof_seedmean.csv',
    'part_C_permutation_sources': ['NB05_wavelength_order_map.csv', 'NB05B_wavelength_order_map.csv'],
    'outer_test_used_for_selection': False, 'predictive_results_modified': False,
    'note': 'Frozen NB01-NB15 outputs are read, never overwritten.'}
(RESULT_DIR / 'NB16_protocol.json').write_text(json.dumps(protocol, indent=2), encoding='utf-8')
(RESULT_DIR / 'NB16_run_summary.json').write_text(json.dumps({
    'status': 'COMPLETED', 'run_revision': RUN_REVISION, 'parts_run': {'A': RUN_PART_A, 'B': RUN_PART_B, 'C': RUN_PART_C},
    'quick_test': QUICK_TEST, 'completed_at_utc': datetime.now(timezone.utc).isoformat()}, indent=2), encoding='utf-8')
with open(RESULT_DIR / 'environment_packages.txt', 'w', encoding='utf-8') as fh:
    fh.write(f'Python: {sys.version}\nPlatform: {platform.platform()}\nTensorFlow: {tf.__version__}\nKeras: {keras.__version__}\n\n')
    try: fh.write(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True, stderr=subprocess.STDOUT))
    except Exception as e: fh.write(f'pip freeze failed: {e}\n')
try: cpuinfo = subprocess.check_output(['lscpu'], text=True, stderr=subprocess.STDOUT)
except Exception as e: cpuinfo = f'lscpu unavailable: {e}'
try: gpuinfo = subprocess.check_output(['nvidia-smi'], text=True, stderr=subprocess.STDOUT)
except Exception as e: gpuinfo = f'nvidia-smi unavailable: {e}'
(RESULT_DIR / 'hardware_info.txt').write_text(cpuinfo + '\n\n' + gpuinfo, encoding='utf-8')

ZIP_NAME = 'NB16_RESULTS_CNN1D_POSITION_AWARE_SENSITIVITY' + ('_QUICKTEST' if QUICK_TEST else '')
zip_base = ZIP_DIR / ZIP_NAME
if zip_base.with_suffix('.zip').exists(): zip_base.with_suffix('.zip').unlink()
tmp = RESULT_DIR.parent / '_NB16_zip_staging'
if tmp.exists(): shutil.rmtree(tmp)
shutil.copytree(RESULT_DIR, tmp, ignore=shutil.ignore_patterns('_CHECKPOINT'))
shutil.make_archive(str(zip_base), 'zip', root_dir=tmp)
shutil.rmtree(tmp)
print('ZIP created:', zip_base.with_suffix('.zip'), '|', round(zip_base.with_suffix('.zip').stat().st_size / 1024, 1), 'KB')
print('NB16 SUCCESS')
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(zip_base.with_suffix('.zip')))